# Train Individual ARIMA Models for Each Region/Service/Target

This notebook trains separate ARIMA models for each combination of:
- Region (eastus, westus, northeurope, southeastasia)
- Service (Container, Storage, VM)
- Target (usage_cpu, usage_storage)

Models are saved with metadata for production deployment.

In [1]:
import pandas as pd
import numpy as np
import pickle
import joblib
import json
from pathlib import Path
from datetime import datetime
import warnings
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore')

def mape(y_true, y_pred):
    """Calculate Mean Absolute Percentage Error"""
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

In [2]:
# Load feature engineered data
df = pd.read_csv('../data/processed/feature_engineered.csv', parse_dates=['date'])

# Check unique combinations
combinations = df[['region', 'resource_type']].drop_duplicates()
targets = ['usage_cpu', 'usage_storage']

print(f"Found {len(combinations)} region/service combinations:")
print(combinations.sort_values(['region', 'resource_type']))
print(f"\nTargets: {targets}")
print(f"Total models to train: {len(combinations) * len(targets)}")

Found 12 region/service combinations:
            region resource_type
0           eastus     Container
90          eastus       Storage
180         eastus            VM
270    northeurope     Container
360    northeurope       Storage
450    northeurope            VM
540  southeastasia     Container
630  southeastasia       Storage
720  southeastasia            VM
810         westus     Container
900         westus       Storage
990         westus            VM

Targets: ['usage_cpu', 'usage_storage']
Total models to train: 24


In [3]:
def train_arima_for_combination(data, region, service, target):
    """Train ARIMA model for specific region/service/target combination"""
    print(f"Training ARIMA for {region}/{service}/{target}...")
    
    # Prepare time series
    ts_data = data.sort_values('date').set_index('date')[target]
    ts_data = ts_data.asfreq('D').fillna(method='ffill')
    
    # Check if we have enough data
    if len(ts_data) < 30:
        print(f"  Insufficient data ({len(ts_data)} points), skipping...")
        return None
    
    # Split for training and validation
    split_point = int(len(ts_data) * 0.8)
    train_data = ts_data[:split_point]
    test_data = ts_data[split_point:]
    
    if len(test_data) < 5:
        train_data = ts_data
        test_data = None
    
    try:
        # Fit ARIMA model (trying different orders)
        best_model = None
        best_aic = float('inf')
        best_order = None
        
        # Try different ARIMA orders
        orders_to_try = [(1,1,1), (2,1,1), (1,1,2), (2,1,2), (0,1,1), (1,0,1)]
        
        for order in orders_to_try:
            try:
                model = ARIMA(train_data, order=order)
                fitted_model = model.fit()
                
                if fitted_model.aic < best_aic:
                    best_aic = fitted_model.aic
                    best_model = fitted_model
                    best_order = order
            except:
                continue
        
        if best_model is None:
            print(f"  Failed to fit any ARIMA model, skipping...")
            return None
        
        # Calculate metrics if we have test data
        metrics = {}
        if test_data is not None and len(test_data) > 0:
            try:
                forecast_result = best_model.get_forecast(steps=len(test_data))
                forecast = forecast_result.predicted_mean
                
                mae = mean_absolute_error(test_data, forecast)
                rmse = np.sqrt(mean_squared_error(test_data, forecast))
                mape_val = mape(test_data, forecast)
                
                metrics = {
                    'mae': float(mae),
                    'rmse': float(rmse),
                    'mape': float(mape_val),
                    'aic': float(best_aic),
                    'test_size': len(test_data)
                }
            except Exception as e:
                print(f"  Warning: Could not calculate validation metrics: {e}")
                metrics = {'aic': float(best_aic)}
        else:
            metrics = {'aic': float(best_aic)}
        
        # Prepare model metadata
        metadata = {
            'region': region,
            'service': service,
            'target': target,
            'model_type': 'ARIMA',
            'model_order': best_order,
            'trained_on': datetime.now().isoformat(),
            'model_version': '1.0',
            'train_size': len(train_data),
            'metrics': metrics,
            'data_range': {
                'start_date': str(train_data.index[0].date()),
                'end_date': str(train_data.index[-1].date())
            }
        }
        
        print(f"  ✓ AIC: {best_aic:.2f}, Order: {best_order}")
        if 'mae' in metrics:
            print(f"  ✓ Validation MAE: {metrics['mae']:.2f}, RMSE: {metrics['rmse']:.2f}")
        
        return best_model, metadata
        
    except Exception as e:
        print(f"  ✗ Error training model: {e}")
        return None

In [4]:
# Create models directory
models_dir = Path('../models/arima')
models_dir.mkdir(parents=True, exist_ok=True)

# Train models for each combination
all_metadata = []
trained_models = {}

for _, row in combinations.iterrows():
    region = row['region']
    service = row['resource_type']
    
    # Filter data for this combination
    combo_data = df[(df['region'] == region) & (df['resource_type'] == service)].copy()
    print(f"\n=== {region} / {service} ===")
    print(f"Data points: {len(combo_data)}")
    
    # Train models for both CPU and Storage
    for target in targets:
        result = train_arima_for_combination(combo_data, region, service, target)
        
        if result:
            model, metadata = result
            
            # Save model and metadata
            model_filename = f"{region}__{service}__{target}_arima.pkl"
            metadata_filename = f"{region}__{service}__{target}_arima_metadata.json"
            
            model_path = models_dir / model_filename
            metadata_path = models_dir / metadata_filename
            
            # Save the fitted model
            joblib.dump(model, model_path)
            
            # Save metadata
            with open(metadata_path, 'w') as f:
                json.dump(metadata, f, indent=2)
            
            print(f"  ✓ Saved: {model_filename}")
            
            # Store for summary
            all_metadata.append(metadata)
            trained_models[(region, service, target)] = model

print(f"\n=== Training Summary ===")
print(f"Successfully trained {len(all_metadata)} models")


=== eastus / Container ===
Data points: 90
Training ARIMA for eastus/Container/usage_cpu...
  ✓ AIC: 586.94, Order: (1, 1, 2)
  ✓ Validation MAE: 13.31, RMSE: 15.13
  ✓ Saved: eastus__Container__usage_cpu_arima.pkl
Training ARIMA for eastus/Container/usage_storage...
  ✓ AIC: 586.94, Order: (1, 1, 2)
  ✓ Validation MAE: 13.31, RMSE: 15.13
  ✓ Saved: eastus__Container__usage_cpu_arima.pkl
Training ARIMA for eastus/Container/usage_storage...
  ✓ AIC: 1080.62, Order: (0, 1, 1)
  ✓ Validation MAE: 460.79, RMSE: 495.35
  ✓ Saved: eastus__Container__usage_storage_arima.pkl

=== eastus / Storage ===
Data points: 90
Training ARIMA for eastus/Storage/usage_cpu...
  ✓ AIC: 1080.62, Order: (0, 1, 1)
  ✓ Validation MAE: 460.79, RMSE: 495.35
  ✓ Saved: eastus__Container__usage_storage_arima.pkl

=== eastus / Storage ===
Data points: 90
Training ARIMA for eastus/Storage/usage_cpu...
  ✓ AIC: 573.09, Order: (0, 1, 1)
  ✓ Validation MAE: 11.73, RMSE: 14.15
  ✓ Saved: eastus__Storage__usage_cpu_arima.

In [5]:
# Save summary metadata
summary_path = models_dir / 'models_summary.json'
summary = {
    'training_date': datetime.now().isoformat(),
    'total_models': len(all_metadata),
    'models': all_metadata
}

with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"Summary saved to: {summary_path}")

# Display model performance summary
print("\n=== Model Performance Summary ===")
for metadata in all_metadata:
    region = metadata['region']
    service = metadata['service']
    target = metadata['target']
    metrics = metadata['metrics']
    
    print(f"{region}/{service}/{target}:")
    print(f"  AIC: {metrics.get('aic', 'N/A'):.2f}")
    if 'mae' in metrics:
        print(f"  MAE: {metrics['mae']:.2f}, RMSE: {metrics['rmse']:.2f}, MAPE: {metrics['mape']:.2f}%")
    print()

Summary saved to: ..\models\arima\models_summary.json

=== Model Performance Summary ===
eastus/Container/usage_cpu:
  AIC: 586.94
  MAE: 13.31, RMSE: 15.13, MAPE: 20.38%

eastus/Container/usage_storage:
  AIC: 1080.62
  MAE: 460.79, RMSE: 495.35, MAPE: 34.60%

eastus/Storage/usage_cpu:
  AIC: 573.09
  MAE: 11.73, RMSE: 14.15, MAPE: 15.99%

eastus/Storage/usage_storage:
  AIC: 1076.12
  MAE: 421.93, RMSE: 489.34, MAPE: 30.37%

eastus/VM/usage_cpu:
  AIC: 575.42
  MAE: 11.12, RMSE: 13.67, MAPE: 16.19%

eastus/VM/usage_storage:
  AIC: 1079.46
  MAE: 370.83, RMSE: 433.28, MAPE: 40.11%

northeurope/Container/usage_cpu:
  AIC: 587.15
  MAE: 12.41, RMSE: 14.74, MAPE: 17.67%

northeurope/Container/usage_storage:
  AIC: 1066.48
  MAE: 413.51, RMSE: 524.24, MAPE: 29.48%

northeurope/Storage/usage_cpu:
  AIC: 591.18
  MAE: 16.76, RMSE: 17.93, MAPE: 21.48%

northeurope/Storage/usage_storage:
  AIC: 1075.54
  MAE: 423.84, RMSE: 485.92, MAPE: 50.11%

northeurope/VM/usage_cpu:
  AIC: 590.47
  MAE: 1